# Session 1: Teach a Model to Read Complaints

Every year, people send millions of complaints about banks, lenders, and payment apps to
the Consumer Financial Protection Bureau (CFPB). Someone has to sort each one: is this
about a credit report? A mortgage? A money transfer that never arrived?

Today you will **fine-tune** a small open language model so it does that sorting for
you, on one GPU of a supercomputer, and then measure how good it got.

**How to run this notebook:** click a cell and press **Shift + Enter**. Run them in
order, top to bottom. A cell is done when the `[*]` beside it turns into a number.

| Step | What you do | About |
|---|---|---|
| 0 | Check you're on a GPU | 2 min |
| 1 | Meet the data | 10 min |
| 2 | Score the model before training | 5 min |
| 3 | Fine-tune it | 15 min |
| 4 | Score it again | 5 min |
| 5 | Try it on your own complaint | 10 min |
| 6 | Your experiment | 30 min |

## 0. Check you're on a GPU

This cell points the notebook at the shared workshop folder and asks the machine what
GPU you have. You should see one NVIDIA GPU with about 96 GB of memory. A laptop
graphics card usually has 4 to 8.

In [ ]:
import json, os, pathlib, subprocess
from IPython.display import Image, display

SHARED = "/work/10539/ashleyscruse/vista/inspire-shared"   # shared data, model, and Python
PY     = f"{SHARED}/venv/bin/python"                       # the workshop's Python
REPO   = str(pathlib.Path.cwd().parent)                    # this repository
RUNS   = f"{os.environ['SCRATCH']}/inspire"                # where your results go

os.environ.update(SHARED=SHARED, PY=PY)
os.environ["LD_LIBRARY_PATH"] = "/opt/apps/gcc14/cuda12/python3/3.11.8/lib:" + os.environ.get("LD_LIBRARY_PATH", "")   # lets the shared Python find libpython
assert pathlib.Path(PY).exists(), "Can't see the shared folder. Raise your hand."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Meet the data

The full CFPB database has about 10 million complaints. We pulled a balanced sample:
the same number of complaints for each of six categories, so the model can't score well
by always guessing the most common one.

Run the cell a few times to read different complaints. Names, account numbers, and
dates were blacked out by the CFPB, which is why you'll see `XXXX`.

In [ ]:
!{PY} {REPO}/scripts/peek.py --shared {SHARED} --n 3

**Think about it:** if you had to sort these by hand, which two categories would you
mix up most often?

## 2. Score the model before training

Our model is **Qwen2.5-0.5B**, an open language model with about half a billion
weights. It has read a lot of the internet, but it has never seen our six categories.
We bolt a new, untrained "pick one of six" layer onto it and test it on 3,000
complaints it will never train on.

With six categories, random guessing gets about **17%** right (1 in 6). Let's see.

You'll see a note that `score.weight` is MISSING. That's expected: it's the new layer,
and it hasn't learned anything yet.

In [ ]:
!{PY} {REPO}/scripts/train.py --shared {SHARED} --out {RUNS}/before --epochs 0

## 3. Fine-tune it

Fine-tuning means showing the model labeled examples and nudging its weights after
each batch so it gets a little less wrong.

We use a technique called **LoRA**. Instead of changing all half-billion weights, we
freeze them and learn a small set of new ones on top: well under 1% of the model. That
is why this fits in a few minutes on one GPU instead of days.

Watch the **loss** as it runs. It measures how wrong the model is, so it should drop.

In [ ]:
TRAIN_EXAMPLES = 6000     # how many labeled complaints to learn from

!{PY} {REPO}/scripts/train.py --shared {SHARED} --out {RUNS}/run1 --limit {TRAIN_EXAMPLES} --save-adapter

## 4. Score it again

Same 3,000 test complaints, before and after. Which categories did it learn best?

In [ ]:
before = json.loads(pathlib.Path(f"{RUNS}/before/results.json").read_text())
after  = json.loads(pathlib.Path(f"{RUNS}/run1/results.json").read_text())

print(f"{'':28} {'before':>8} {'after':>8}")
print(f"{'ALL CATEGORIES':28} {before['accuracy']:8.0%} {after['accuracy']:8.0%}\n")
for name in after["per_class"]:
    print(f"{name:28} {before['per_class'][name]:8.0%} {after['per_class'][name]:8.0%}")
print(f"\nTraining took {after['train_seconds']:.0f} seconds on 1 GPU "
      f"({after['examples_per_second']:.0f} complaints per second).")

**Write these down. You'll need them this afternoon:**

- Accuracy after training: ______
- Seconds it took to train: ______

## 5. Try it on your own complaint

Write a complaint the way a real person would. Try an easy one, then try to trick it:
a complaint that could fit two categories.

In [ ]:
MY_COMPLAINTS = [
    "I sent $200 to my brother through a payment app and it never showed up. Customer service keeps closing my ticket.",
    "A collector keeps calling my job about a medical bill I already paid.",
    "Write your own here.",
]

cmd = [PY, f"{REPO}/scripts/predict.py", "--shared", SHARED, "--adapter", f"{RUNS}/run1/adapter"]
for c in MY_COMPLAINTS:
    cmd += ["--text", c]
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

## 6. Your experiment

Pick **one** question, change **one** number in the cell below, run it, and fill in the
table. Use a new `RUN_NAME` each time so you don't overwrite earlier results.

1. **More data:** does accuracy keep climbing from 600 to 6,000 to 24,000 examples?
   What happens to training time?
2. **Shorter reading:** `MAX_LEN` is how many tokens (word pieces) of each complaint the
   model reads. Try 64 and 512. Is the first part of a complaint enough?
3. **More passes:** `EPOCHS = 2` shows the model every example twice. Better, or just
   slower?

| Run | What I changed | Accuracy | Seconds |
|---|---|---|---|
| run1 | nothing (6,000 examples) | | |
| | | | |
| | | | |

In [ ]:
RUN_NAME       = "exp1"
TRAIN_EXAMPLES = 24000
MAX_LEN        = 256
EPOCHS         = 1

!{PY} {REPO}/scripts/train.py --shared {SHARED} --out {RUNS}/{RUN_NAME} --limit {TRAIN_EXAMPLES} --max-len {MAX_LEN} --epochs {EPOCHS}

## What you just did

- Ran a language model on a supercomputer GPU.
- Measured it before training, so the "after" number means something.
- Fine-tuned it with LoRA in minutes.
- Ran a controlled experiment: one change, one measurement.

**Before you go to lunch:** close this session so its node goes back to the pool for
this afternoon. Your results are saved in your scratch folder.

**This afternoon:** you probably noticed that more data meant better accuracy and
longer waits. The full database has millions of complaints. In *Make It Faster*, your
team will spread one training run across several GPUs at once and measure how much
faster it gets.